# A classification report in NumPy

scikit-learn's `classification_report` prints four numbers per class (precision, recall,
F1 and support), then the accuracy and the macro and weighted averages of F1 and the rest. Here you compute
the same numbers from two label arrays with NumPy alone, on twelve photos of three animals.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Twelve photos, three classes: 0 cat, 1 dog, 2 fox.
y_true = np.array([0, 1, 2, 0, 0, 1, 2, 0, 1, 2, 0, 1])
y_pred = np.array([0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1])
print("true:     ", y_true)
print("predicted:", y_pred)

## Part 1 (coding task): The confusion matrix

Write `confusion_matrix(y_true, y_pred, k)` for labels $0, \dots, k - 1$. It returns a
$k \times k$ array of integers whose entry `[i, j]` counts the examples of true class $i$
that were predicted as class $j$: true classes on rows, predictions on columns, the layout
scikit-learn uses.

A hint, if you want one: the pair $(i, j)$ can be numbered $k i + j$, and `np.bincount`
counts how often each number occurs.

In [ ]:
def confusion_matrix(y_true, y_pred, k):
    """k x k counts: row i, column j holds the examples of class i predicted as j."""
    # TODO
    raise NotImplementedError

In [ ]:
C = confusion_matrix(y_true, y_pred, 3)
print(C)
assert C.shape == (3, 3), f"expected shape (3, 3), got {C.shape}"
assert np.issubdtype(C.dtype, np.integer), "the entries should be integer counts"
assert C.sum() == len(y_true), "every example should be counted exactly once"
expected = np.array([[4, 1, 0], [1, 3, 0], [2, 1, 0]])
assert not np.array_equal(C, expected.T), "rows and columns are swapped: rows must be the true classes"
assert np.array_equal(C, expected), "some counts are wrong"
only_dogs = confusion_matrix(np.array([1, 1]), np.array([1, 1]), 3)
assert only_dogs.shape == (3, 3) and only_dogs[1, 1] == 2, "classes that never occur still need a row and a column"
print("Part 1 passed")

## Part 2 (coding task): Per-class scores and the three averages

Write `report(C)`, which takes a confusion matrix in that layout and returns a dictionary with

- `"precision"`, `"recall"`, `"f1"` and `"support"`: arrays with one entry per class.
- `"accuracy"`, `"macro_f1"` and `"weighted_f1"`: single numbers.

A class's precision is its diagonal entry over its column sum, its recall the diagonal
entry over its row sum, and its support the row sum. Compute F1 as
$2TP / (2TP + FP + FN)$, which is 0 for a class with no true positives. A class that is
never predicted has no precision to speak of: report 0, as scikit-learn does (with a
warning). The macro F1 is the plain mean of the class F1 scores, and the weighted F1
weights each one by its support.

In [ ]:
def report(C):
    C = np.asarray(C, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
r = report(np.array([[4, 1, 0], [1, 3, 0], [2, 1, 0]]))
for key in ["precision", "recall", "f1", "support", "accuracy", "macro_f1", "weighted_f1"]:
    assert key in r, f"the result has no {key!r}"
assert not np.isnan(r["precision"]).any(), "the fox class is never predicted: its precision should be 0, not nan"
assert np.allclose(r["precision"], [4 / 7, 3 / 5, 0]), f"precision {r['precision']}: divide the diagonal by the column sums"
assert np.allclose(r["recall"], [4 / 5, 3 / 4, 0]), f"recall {r['recall']}: divide the diagonal by the row sums"
assert np.allclose(r["support"], [5, 4, 3]), f"support {r['support']}: the row sums"
assert np.allclose(r["f1"], [2 / 3, 2 / 3, 0]), f"f1 {r['f1']}"
assert np.isclose(r["accuracy"], 7 / 12), f"accuracy {r['accuracy']}"
assert np.isclose(r["macro_f1"], 4 / 9), f"macro_f1 {r['macro_f1']}: the plain mean of the three class F1 scores"
assert np.isclose(r["weighted_f1"], 1 / 2), f"weighted_f1 {r['weighted_f1']}: weight each class F1 by its support"
perfect = report(np.diag([3, 5, 2]))
assert np.allclose(perfect["f1"], 1) and np.isclose(perfect["macro_f1"], 1), "a perfect matrix scores 1 everywhere"
print("Part 2 passed")

## Solution

### Part 1: The confusion matrix

In [ ]:
def confusion_matrix(y_true, y_pred, k):
    """k x k counts: row i, column j holds the examples of class i predicted as j."""
    cells = k * np.asarray(y_true) + np.asarray(y_pred)
    return np.bincount(cells, minlength=k * k).reshape(k, k)

**Why it works.** Every (true, predicted) pair names one cell, and $k i + j$ numbers the cells row by row
from $0$ to $k^2 - 1$. `np.bincount` counts every cell in one pass, and `reshape(k, k)`
puts cell $k i + j$ back at row $i$, column $j$. `minlength=k * k` keeps the matrix
$k \times k$ when the last classes never occur, as in the all-dog check.

### Part 2: Per-class scores and the three averages

In [ ]:
def report(C):
    C = np.asarray(C, dtype=float)
    tp = np.diag(C)
    predicted = C.sum(axis=0)          # TP + FP for each class
    support = C.sum(axis=1)            # TP + FN for each class

    def ratio(top, bottom):
        return np.divide(top, bottom, out=np.zeros_like(top), where=bottom > 0)

    f1 = ratio(2 * tp, predicted + support)   # 2TP + FP + FN = column sum + row sum
    return {
        "precision": ratio(tp, predicted),
        "recall": ratio(tp, support),
        "f1": f1,
        "support": support,
        "accuracy": tp.sum() / C.sum(),
        "macro_f1": f1.mean(),
        "weighted_f1": (f1 * support).sum() / support.sum(),
    }

**Why it works.** Column $j$ holds everything predicted as $j$, so its sum is $TP + FP$ for class $j$, and
row $i$ holds everything that truly is $i$, so its sum is $TP + FN$. Their total is
$2TP + FP + FN$, the F1 denominator, so F1 needs neither precision nor recall and never
divides 0 by 0 unless a class has no rows and no predictions at all. On the photos the
averages separate: accuracy $7/12 \approx 0.583$, weighted F1 $1/2$, macro F1
$4/9 \approx 0.444$. The fox class, never predicted, has a quarter of the rows and a
third of the macro vote, so it pulls the macro average down the most.